# 03 - Feature Engineering

### Notebook idea:
- zbudować cechy na bazie danych OHLCV,
- zachować brak leakage (cechy tylko z historii do chwili t),
- przypisać rekordy do istniejących splitów z notebooka 02 (bez tworzenia nowego splitu),
- zapisać zbiory cech do dalszych analiz

## Mapping to Lab6/7 Points
- **Punkt 2**: feature engineering na danych czasowych z kontrol? leakage (rolling/lag), przygotowanie cech do dalszej selekcji i modelowania.


In [11]:
import os

import numpy as np
import pandas as pd
from IPython.display import display
from stylin import InfoDisplayStyler


In [12]:
styler = InfoDisplayStyler()

In [13]:
DATA_PATH = '../../data/raw/ethusdt_1h.csv'
CLEAN_PATH = '../../data/clean/'
LABELED_PATH = '../../data/labeled/'
PROCESSED_PATH = '../../data/processed/'

raw_data = pd.read_csv(DATA_PATH)
assert not raw_data.empty, "DataFrame is empty"

os.makedirs(CLEAN_PATH, exist_ok=True)
os.makedirs(LABELED_PATH, exist_ok=True)
os.makedirs(PROCESSED_PATH, exist_ok=True)

LABELED_FILE = LABELED_PATH + 'eth_labeled.csv'
TRAIN_SPLIT_FILE = LABELED_PATH + 'train_labeled.csv'
VAL_SPLIT_FILE = LABELED_PATH + 'val_labeled.csv'
TEST_SPLIT_FILE = LABELED_PATH + 'test_labeled.csv'

assert os.path.exists(LABELED_FILE), 'Brak eth_labeled.csv. Najpierw uruchom notebook 01.'
assert os.path.exists(TRAIN_SPLIT_FILE), 'Brak train_labeled.csv. Najpierw uruchom notebook 02.'
assert os.path.exists(VAL_SPLIT_FILE), 'Brak val_labeled.csv. Najpierw uruchom notebook 02.'
assert os.path.exists(TEST_SPLIT_FILE), 'Brak test_labeled.csv. Najpierw uruchom notebook 02.'

labeled_df = pd.read_csv(LABELED_FILE)
train_split_df = pd.read_csv(TRAIN_SPLIT_FILE)
val_split_df = pd.read_csv(VAL_SPLIT_FILE)
test_split_df = pd.read_csv(TEST_SPLIT_FILE)

styler.show_meta(labeled_df)
styler.style_me(labeled_df.head(5), title='Labeled input preview')


,timestamp,open,high,low,close,volume,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,taker_buy_quote_asset_volume,future_return,target
0,2021-01-01 00:00:00+00:00,736.420000,739.000000,729.330000,734.070000,27932.698840,20479903.476036,22671,15020.616630,11015918.972215,0.008037,2
1,2021-01-01 01:00:00+00:00,734.080000,749.000000,733.370000,748.280000,52336.187790,38899618.853014,41712,27395.902700,20362373.933503,-0.014567,0
2,2021-01-01 02:00:00+00:00,748.270000,749.000000,742.270000,744.060000,33019.501000,24606726.438367,19566,17721.266960,13207123.352084,-0.018802,0
3,2021-01-01 03:00:00+00:00,744.060000,747.230000,743.100000,744.820000,17604.808590,13119088.400414,12230,9499.749200,7079926.310875,-0.014957,0
4,2021-01-01 04:00:00+00:00,744.870000,747.090000,739.300000,742.290000,18794.154240,13980190.971469,13626,10133.742110,7539500.286872,-0.007383,1


### Minimal timestamp preparation

In [14]:
# Minimal preparation: parse timestamp and keep chronological order.
for df in [labeled_df, train_split_df, val_split_df, test_split_df]:
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True, errors='coerce')

labeled_df = labeled_df.sort_values('timestamp').reset_index(drop=True)

split_check = pd.DataFrame({
    'dataset': ['labeled', 'train', 'val', 'test'],
    'rows': [len(labeled_df), len(train_split_df), len(val_split_df), len(test_split_df)],
    'min_ts': [labeled_df['timestamp'].min(), train_split_df['timestamp'].min(), val_split_df['timestamp'].min(), test_split_df['timestamp'].min()],
    'max_ts': [labeled_df['timestamp'].max(), train_split_df['timestamp'].max(), val_split_df['timestamp'].max(), test_split_df['timestamp'].max()],
})
styler.style_me(split_check, title='Input split summary')


,dataset,rows,min_ts,max_ts
0,labeled,46252,2021-01-01 00:00:00+00:00,2026-04-12 17:00:00+00:00
1,train,32376,2021-01-01 00:00:00+00:00,2024-09-11 13:00:00+00:00
2,val,6937,2024-09-11 14:00:00+00:00,2025-06-27 14:00:00+00:00
3,test,6939,2025-06-27 15:00:00+00:00,2026-04-12 17:00:00+00:00


### Feature engineering function definitions

### Opis tworzonych cech

| Nazwa cechy / grupa | Z czego powstaje (jak liczona) | Co wnosi do modelu |
|---|---|---|
| `return_1`, `return_2`, `return_3`, `return_6`, `return_12`, `return_24` | `close.pct_change(period)` | Informacja o kierunku i sile zmiany ceny w różnych horyzontach czasowych. |
| `log_return_1`, `log_return_6`, `log_return_24` | `np.log(close).diff(period)` | Bardziej stabilna numerycznie wersja zwrotu, przydatna przy analizie zmienności. |
| `close_open_pct` | `(close - open) / open` | Kierunek i siła świecy (czy godzina zakończyła się wzrostem czy spadkiem). |
| `high_low_pct` | `(high - low) / close` | Szerokość świecy, czyli intrahour zmienność. |
| `upper_shadow` | `(high - max(open, close)) / close` | Wielkość górnego knota, sygnał odrzucenia wyższych cen. |
| `lower_shadow` | `(min(open, close) - low) / close` | Wielkość dolnego knota, sygnał odrzucenia niższych cen. |
| `body_size` | `abs(close - open) / close` | Siła korpusu świecy, proxy impetu ruchu w godzinie. |
| `rolling_close_mean_*` | `close.rolling(window).mean()` | Lokalny poziom trendu ceny w oknach czasowych. |
| `rolling_return_std_*` | `return_1.rolling(window).std()` | Lokalna zmienność krótkoterminowa. |
| `rolling_close_min_*`, `rolling_close_max_*` | `close.rolling(window).min()/max()` | Lokalny zakres cen, poziomy skrajne. |
| `rolling_return_skew_*` | `return_1.rolling(window).skew()` | Asymetria rozkładu zwrotów (przewaga dodatnich/ujemnych ogonów). |
| `rolling_return_kurt_*` | `return_1.rolling(window).kurt()` | „Grubość ogonów” rozkładu zwrotów (ryzyko skoków). |
| `roc_*` | `close.pct_change(period)` | Tempo zmian ceny (Rate of Change) w danym horyzoncie. |
| `momentum_*` | `close - close.shift(period)` | Surowa różnica ceny, sygnał impetu trendu. |
| `sma_*` | `close.rolling(period).mean()` | Wygładzony trend krótkoterminowy/średnioterminowy. |
| `ema_*` | `close.ewm(span=period).mean()` | Średnia ważona, szybciej reaguje na nowe dane niż SMA. |
| `dist_sma_*` | `(close - sma_*) / close` | Odchylenie ceny od SMA (czy cena jest „nad” lub „pod” trendem). |
| `dist_ema_*` | `(close - ema_*) / close` | Odchylenie ceny od EMA (szybszy sygnał niż dist_sma). |
| `rsi_14` | RSI z 14 okresów (średnie wzrosty/spadki) | Siła ruchu i potencjalne stany wykupienia/wyprzedania. |
| `macd`, `macd_signal`, `macd_hist` | EMA(12), EMA(26), EMA sygnału(9) | Informacja o zmianie dynamiki trendu i przecięciach momentum. |
| `atr_14` | True Range + średnia z 14 okresów | Miara zmienności niezależna od kierunku ruchu. |
| `bb_width` | Szerokość Bollinger Bands względem średniej | Stopień kompresji/rozszerzenia zmienności. |
| `bb_position` | Pozycja ceny względem Bollinger Bands | Gdzie cena leży w kanale zmienności (góra/środek/dół). |
| `volume_return` | `volume.pct_change(1)` | Nagłe zmiany aktywności wolumenowej. |
| `volume_mean_24` | `volume.rolling(24).mean()` | Bazowy poziom wolumenu dobowego. |
| `volume_std_24` | `volume.rolling(24).std()` | Zmienność wolumenu. |
| `volume_zscore_24` | `(volume - volume_mean_24) / volume_std_24` | Anomalie wolumenu względem lokalnej normy. |
| `hour_sin`, `hour_cos` | Sin/cos z godziny doby | Cykliczność dobowa bez sztucznych skoków między 23 i 0. |
| `dow_sin`, `dow_cos` | Sin/cos z dnia tygodnia | Cykliczność tygodniowa handlu. |
| `is_weekend` | `dayofweek >= 5` | Informacja o weekendzie (inna płynność/charakter rynku). |
| `return_1_lag_*`, `return_6_lag_*`, `rsi_14_lag_*`, `macd_hist_lag_*`, `volume_zscore_24_lag_*` | `feature.shift(lag)` dla lagów 1/2/3 | Krótkoterminowa pamięć sygnału i opóźnione zależności czasowe. |


- RSI = Relative Strength Index
- MACD = Moving Average Convergence Divergence
-  ATR = Average True Range

In [ ]:
# Oblicza wskaźnik RSI na podstawie średnich wykładniczych wzrostów i spadków ceny z zadanego okresu.
def _rsi(series: pd.Series, period: int = 14) -> pd.Series:
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / period, min_periods=period).mean()
    avg_loss = loss.ewm(alpha=1 / period, min_periods=period).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

# Wylicza MACD jako różnicę EMA szybkiej i wolnej oraz zwraca linię sygnału i histogram.
def _macd(series: pd.Series) -> tuple[pd.Series, pd.Series, pd.Series]:
    ema_fast = series.ewm(span=12, adjust=False).mean()
    ema_slow = series.ewm(span=26, adjust=False).mean()
    macd = ema_fast - ema_slow
    signal = macd.ewm(span=9, adjust=False).mean()
    hist = macd - signal
    return macd, signal, hist

# Oblicza ATR jako średnią kroczącą True Range, czyli miarę zmienności rynku.
def _atr(df: pd.DataFrame, period: int = 14) -> pd.Series:
    high_low = df['high'] - df['low']
    high_close = (df['high'] - df['close'].shift(1)).abs()
    low_close = (df['low'] - df['close'].shift(1)).abs()
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    return tr.rolling(period).mean()


def create_features(df: pd.DataFrame) -> pd.DataFrame:
    work = df.copy()

    # short-term micro-moves: 1-3h, intraday context: 6-12h, daily context: 24h
    for period in [1, 2, 3, 6, 12, 24]:
        work[f'return_{period}'] = work['close'].pct_change(period)

    work['log_return_1'] = np.log(work['close']).diff(1)
    work['log_return_6'] = np.log(work['close']).diff(6)
    work['log_return_24'] = np.log(work['close']).diff(24)

    work['close_open_pct'] = (work['close'] - work['open']) / work['open']
    work['high_low_pct'] = (work['high'] - work['low']) / work['close']
    work['upper_shadow'] = (work['high'] - work[['open', 'close']].max(axis=1)) / work['close']
    work['lower_shadow'] = (work[['open', 'close']].min(axis=1) - work['low']) / work['close']
    work['body_size'] = (work['close'] - work['open']).abs() / work['close']

    # 6h and 12h capture intraday behavior, 24h and 48h capture daily / 2-day structure
    for window in [6, 12, 24, 48]:
        work[f'rolling_close_mean_{window}'] = work['close'].rolling(window).mean()
        work[f'rolling_return_std_{window}'] = work['return_1'].rolling(window).std()
        work[f'rolling_close_min_{window}'] = work['close'].rolling(window).min()
        work[f'rolling_close_max_{window}'] = work['close'].rolling(window).max()
        work[f'rolling_return_skew_{window}'] = work['return_1'].rolling(window).skew()
        work[f'rolling_return_kurt_{window}'] = work['return_1'].rolling(window).kurt()

    for period in [6, 12, 24]:
        work[f'roc_{period}'] = work['close'].pct_change(period)
        work[f'momentum_{period}'] = work['close'] - work['close'].shift(period)
        work[f'sma_{period}'] = work['close'].rolling(period).mean()
        work[f'ema_{period}'] = work['close'].ewm(span=period, adjust=False).mean()
        work[f'dist_sma_{period}'] = (work['close'] - work[f'sma_{period}']) / work['close']
        work[f'dist_ema_{period}'] = (work['close'] - work[f'ema_{period}']) / work['close']

    work['rsi_14'] = _rsi(work['close'], period=14)
    macd, macd_signal, macd_hist = _macd(work['close'])
    work['macd'] = macd
    work['macd_signal'] = macd_signal
    work['macd_hist'] = macd_hist
    work['atr_14'] = _atr(work, period=14)

    rolling_mean_20 = work['close'].rolling(20).mean()
    rolling_std_20 = work['close'].rolling(20).std()
    bb_upper = rolling_mean_20 + 2 * rolling_std_20
    bb_lower = rolling_mean_20 - 2 * rolling_std_20
    work['bb_width'] = (bb_upper - bb_lower) / rolling_mean_20
    work['bb_position'] = (work['close'] - bb_lower) / (bb_upper - bb_lower)

    work['volume_return'] = work['volume'].pct_change(1)
    work['volume_mean_24'] = work['volume'].rolling(24).mean()
    work['volume_std_24'] = work['volume'].rolling(24).std()
    work['volume_zscore_24'] = (work['volume'] - work['volume_mean_24']) / work['volume_std_24']

    if not isinstance(work.index, pd.DatetimeIndex):
        raise ValueError('DataFrame index must be DatetimeIndex for calendar features.')

    hour = work.index.hour
    day = work.index.dayofweek
    work['hour_sin'] = np.sin(2 * np.pi * hour / 24)
    work['hour_cos'] = np.cos(2 * np.pi * hour / 24)
    work['dow_sin'] = np.sin(2 * np.pi * day / 7)
    work['dow_cos'] = np.cos(2 * np.pi * day / 7)
    work['is_weekend'] = (day >= 5).astype(int)

    lag_features = ['return_1', 'return_6', 'rsi_14', 'macd_hist', 'volume_zscore_24']
    for col in lag_features:
        for lag in [1, 2, 3]:
            work[f'{col}_lag_{lag}'] = work[col].shift(lag)

    return work


### Feature construction and leakage control
- z danych bazowych powstaje macierz cech do modelowania.

In [ ]:
work_df = labeled_df.copy()
work_df = work_df.sort_values('timestamp').set_index('timestamp')

features_full = create_features(work_df)

assert 'target' in features_full.columns
assert 'future_return' in features_full.columns

feature_cols = [c for c in features_full.columns if c not in ['target', 'future_return']]
for forbidden in ['lead', 'future_target', 'target_shift_minus']:
    assert not any(forbidden in c for c in feature_cols), f'Wykryto podejrzana kolumne: {forbidden}'

styler.show_line(len(feature_cols), title='Number of candidate feature columns')
styler.style_me(pd.DataFrame({'feature_columns_sample': feature_cols[:30]}), title='Feature sample (first 30)')


,feature_columns_sample
0,open
1,high
2,low
3,close
4,volume
5,quote_asset_volume
6,number_of_trades
7,taker_buy_base_asset_volume
8,taker_buy_quote_asset_volume
9,return_1


### NaN/Inf report after lags/rolling and cleaning

In [17]:
inf_report = (
    pd.DataFrame(
        {
            'column': features_full.columns,
            'pos_inf_count': np.isinf(features_full.to_numpy(dtype=np.float64)).sum(axis=0),
            'neg_inf_count': np.isneginf(features_full.to_numpy(dtype=np.float64)).sum(axis=0),
        }
    )
)
inf_report['inf_total'] = inf_report['pos_inf_count'] + inf_report['neg_inf_count']
inf_report = inf_report[inf_report['inf_total'] > 0].sort_values('inf_total', ascending=False)

if len(inf_report) > 0:
    styler.style_me(inf_report.head(30), title='Columns with +/-inf after feature engineering')

features_full = features_full.replace([np.inf, -np.inf], np.nan)

nan_report = features_full.isna().sum().sort_values(ascending=False)
nan_report = nan_report[nan_report > 0].to_frame('nan_count').reset_index().rename(columns={'index': 'column'})

styler.style_me(nan_report.head(30), title='Top NaN counts after feature engineering')

rows_before_dropna = len(features_full)
features_clean = features_full.dropna().copy()
rows_after_dropna = len(features_clean)
rows_removed = rows_before_dropna - rows_after_dropna

dropna_report = pd.DataFrame(
    {
        'rows_before_dropna': [rows_before_dropna],
        'rows_after_dropna': [rows_after_dropna],
        'rows_removed': [rows_removed],
        'removed_pct': [round(rows_removed / rows_before_dropna * 100, 4)],
    }
)
styler.style_me(dropna_report, title='Dropna report')


,column,pos_inf_count,neg_inf_count,inf_total
74,volume_return,2,0,2


,column,nan_count
0,rolling_return_kurt_48,48
1,rolling_return_skew_48,48
2,rolling_return_std_48,48
3,rolling_close_mean_48,47
4,rolling_close_max_48,47
5,rolling_close_min_48,47
6,volume_zscore_24_lag_3,26
7,volume_zscore_24_lag_2,25
8,momentum_24,24
9,rolling_return_kurt_24,24


,rows_before_dropna,rows_after_dropna,rows_removed,removed_pct
0,46252,46202,50,0.108100


### Cleaning data thoughts:

Po utworzeniu cech pojawiły się wartości `NaN` i pojedyncze `+inf`, co jest naturalnym efektem operacji sekwencyjnych (rolling, lagi, zmiany procentowe).

- Wykryte `+inf` wystąpiły w kolumnie `volume_return` (2 przypadki).  
  Źródło: `pct_change` przy zerowym wolumenie w poprzedniej obserwacji (dzielenie przez 0).  
  Dlatego zastosowano bezpieczne czyszczenie: `±inf -> NaN`.

- Największe liczby `NaN` występują w cechach opartych o długie okna (`window=48`) i statystyki rolling (`std/skew/kurt`), np. `rolling_return_kurt_48`, `rolling_return_skew_48`, `rolling_return_std_48`.  
  To oczekiwane, ponieważ takie cechy wymagają pełnego „rozbiegu” okna i nie mogą być policzone dla pierwszych obserwacji.

- `NaN` pojawiają się także w cechach lagowanych (np. `*_lag_1/2/3`) oraz w cechach opartych o horyzont 24h (`return_24`, `roc_24`, `momentum_24`, `sma_24`), ponieważ każda z nich potrzebuje odpowiedniej liczby poprzednich punktów.

Po zamianie `inf` na `NaN` i zastosowaniu `dropna`:
- `rows_before_dropna = 46252`
- `rows_after_dropna = 46202`
- `rows_removed = 50` (`~0.1081%`)

Usunięto bardzo mały odsetek rekordów, więc wpływ na reprezentatywność danych jest minimalny, a jakość wejścia do modelu znacząco lepsza (brak wartości niefinitywnych i niekompletnych).

### Assignment to existing splits from notebook 02

In [18]:
train_ts = set(train_split_df['timestamp'])
val_ts = set(val_split_df['timestamp'])
test_ts = set(test_split_df['timestamp'])

features_reset = features_clean.reset_index().copy()

train_features = features_reset[features_reset['timestamp'].isin(train_ts)].copy()
val_features = features_reset[features_reset['timestamp'].isin(val_ts)].copy()
test_features = features_reset[features_reset['timestamp'].isin(test_ts)].copy()

split_overlap_report = pd.DataFrame({
    'check': ['train_val_overlap', 'train_test_overlap', 'val_test_overlap'],
    'value': [
        len(set(train_features['timestamp']).intersection(set(val_features['timestamp']))),
        len(set(train_features['timestamp']).intersection(set(test_features['timestamp']))),
        len(set(val_features['timestamp']).intersection(set(test_features['timestamp']))),
    ]
})
styler.style_me(split_overlap_report, title='Overlap check after feature assignment')

features_split_report = pd.DataFrame({
    'split': ['train_features', 'val_features', 'test_features'],
    'rows': [len(train_features), len(val_features), len(test_features)],
    'min_ts': [train_features['timestamp'].min(), val_features['timestamp'].min(), test_features['timestamp'].min()],
    'max_ts': [train_features['timestamp'].max(), val_features['timestamp'].max(), test_features['timestamp'].max()],
})
styler.style_me(features_split_report, title='Feature split summary')


,check,value
0,train_val_overlap,0
1,train_test_overlap,0
2,val_test_overlap,0


,split,rows,min_ts,max_ts
0,train_features,32326,2021-01-03 00:00:00+00:00,2024-09-11 13:00:00+00:00
1,val_features,6937,2024-09-11 14:00:00+00:00,2025-06-27 14:00:00+00:00
2,test_features,6939,2025-06-27 15:00:00+00:00,2026-04-12 17:00:00+00:00


### Quick feature report

In [19]:
model_feature_cols = [c for c in train_features.columns if c not in ['timestamp', 'target', 'future_return']]

summary_report = pd.DataFrame({
    'metric': ['all_feature_columns', 'model_feature_columns', 'train_rows', 'val_rows', 'test_rows'],
    'value': [
        len(feature_cols),
        len(model_feature_cols),
        len(train_features),
        len(val_features),
        len(test_features),
    ],
})
styler.style_me(summary_report, title='Feature engineering summary')

styler.style_me(train_features[model_feature_cols].describe().T.head(20), title='Train feature stats (sample)')


,metric,value
0,all_feature_columns,96
1,model_feature_columns,96
2,train_rows,32326
3,val_rows,6937
4,test_rows,6939


,count,mean,std,min,25%,50%,75%,max
open,32326.000000,2353.336937,877.026666,772.440000,1651.937500,2080.985000,3060.637500,4846.940000
high,32326.000000,2366.286804,881.873972,778.440000,1658.772500,2090.635000,3078.037500,4868.000000
low,32326.000000,2339.426145,871.633946,768.710000,1645.770000,2070.165000,3040.997500,4833.190000
close,32326.000000,2353.383743,876.983093,772.370000,1651.957500,2081.025000,3060.635000,4846.710000
volume,32326.000000,25142.775613,28012.053162,0.000000,9314.362950,16509.822400,30480.678343,493227.882820
quote_asset_volume,32326.000000,55201283.119651,59016320.137777,0.000000,20818157.588868,38615335.233544,68016878.416895,1170475765.355900
number_of_trades,32326.000000,41249.951680,36384.433980,0.000000,19755.000000,31086.000000,50009.750000,1067023.000000
taker_buy_base_asset_volume,32326.000000,12523.885781,13938.937043,0.000000,4607.357125,8250.416050,15177.926525,238996.465800
taker_buy_quote_asset_volume,32326.000000,27493349.584836,29266344.083677,0.000000,10303611.268760,19292910.179072,33950951.761822,563990279.637153
return_1,32326.000000,0.000072,0.008663,-0.130794,-0.003073,0.000067,0.003298,0.092128


### Saving artifacts

In [20]:
FEATURES_FULL_OUT = PROCESSED_PATH + 'features_full.csv'
TRAIN_FEATURES_OUT = PROCESSED_PATH + 'train_features.csv'
VAL_FEATURES_OUT = PROCESSED_PATH + 'val_features.csv'
TEST_FEATURES_OUT = PROCESSED_PATH + 'test_features.csv'
FEATURE_LIST_OUT = PROCESSED_PATH + 'feature_columns.csv'

features_reset.to_csv(FEATURES_FULL_OUT, index=False)
train_features.to_csv(TRAIN_FEATURES_OUT, index=False)
val_features.to_csv(VAL_FEATURES_OUT, index=False)
test_features.to_csv(TEST_FEATURES_OUT, index=False)

pd.DataFrame({'feature_column': model_feature_cols}).to_csv(FEATURE_LIST_OUT, index=False)

styler.show_line(FEATURES_FULL_OUT, title='Saved full features')
styler.show_line(TRAIN_FEATURES_OUT, title='Saved train features')
styler.show_line(VAL_FEATURES_OUT, title='Saved val features')
styler.show_line(TEST_FEATURES_OUT, title='Saved test features')
styler.show_line(FEATURE_LIST_OUT, title='Saved feature list')


### Final thoughts:

Notebook 03 realizuje etap feature engineeringu i przygotowania danych wejściowych pod modelowanie.  
Na bazie danych OHLCV utworzono szeroki zestaw cech opisujących rynek w różnych horyzontach czasowych: zwroty, cechy świec, statystyki rolling, momentum/trend, wskaźniki techniczne, cechy wolumenowe, kalendarzowe i lagi.

W procesie tworzenia cech pojawiły się oczekiwane `NaN` (rozbieg okien rolling i lagów) oraz incydentalne `+inf` (np. w `volume_return` przy zerowym mianowniku w `pct_change`).  
Zastosowane czyszczenie (`±inf -> NaN -> dropna`) usunęło tylko niewielką część obserwacji (`50` rekordów, ok. `0.1081%`), co nie wpływa istotnie na reprezentatywność zbioru.

Po czyszczeniu rekordy zostały poprawnie przypisane do istniejących splitów z notebooka 02 (bez tworzenia nowego podziału i bez overlapu między splitami), a finalne pliki `train_features/val_features/test_features` zostały zapisane do dalszych etapów.